# House Prices - Kaggle Competition
## Ensemble Methods: Random Forest & Gradient Boosting

Predicción de precios de casas usando métodos de ensamble.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

## 2. Carga de datos

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
train.head()

## 3. Análisis Exploratorio (EDA)

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
# Distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train['SalePrice'], bins=50, edgecolor='black')
axes[0].set_title('Distribución de SalePrice')
axes[0].set_xlabel('SalePrice')

axes[1].hist(np.log1p(train['SalePrice']), bins=50, edgecolor='black')
axes[1].set_title('Distribución de log(SalePrice)')
axes[1].set_xlabel('log(SalePrice)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlación con SalePrice (top 15 features)
numeric_cols = train.select_dtypes(include=[np.number]).columns
correlation = train[numeric_cols].corr()['SalePrice'].abs().sort_values(ascending=False)
top_features = correlation.head(16).index  # 15 + SalePrice

plt.figure(figsize=(12, 10))
sns.heatmap(train[top_features].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlación - Top 15 features vs SalePrice')
plt.tight_layout()
plt.show()

print("Top 15 features correlacionadas con SalePrice:")
print(correlation.head(16))

## 4. Análisis de valores nulos

In [ ]:
# Valores nulos en train y test
null_train = train.isnull().sum()
null_test = test.isnull().sum()

null_df = pd.DataFrame({
    'Train_nulls': null_train[null_train > 0],
    'Train_%': (null_train[null_train > 0] / len(train) * 100).round(2),
    'Test_nulls': null_test.reindex(null_train[null_train > 0].index),
    'Test_%': (null_test.reindex(null_train[null_train > 0].index) / len(test) * 100).round(2)
}).sort_values('Train_%', ascending=False)

print(f"Columnas con nulos en train: {null_df.shape[0]}")
null_df

## 5. Preprocesamiento

Combinamos train y test para aplicar las mismas transformaciones.

In [ ]:
# Guardar IDs y target
train_ids = train['Id']
test_ids = test['Id']
y = np.log1p(train['SalePrice'])  # Transformación log para normalizar

# Eliminar Id y SalePrice antes de combinar
train_features = train.drop(['Id', 'SalePrice'], axis=1)
test_features = test.drop(['Id'], axis=1)

# Combinar para preprocesar juntos
ntrain = train_features.shape[0]
all_data = pd.concat([train_features, test_features], axis=0, ignore_index=True)
print(f"All data shape: {all_data.shape}")

In [ ]:
# Eliminar columnas con demasiados nulos (>80%)
high_null_cols = [col for col in all_data.columns if all_data[col].isnull().sum() / len(all_data) > 0.80]
print(f"Columnas eliminadas por >80% nulos: {high_null_cols}")
all_data = all_data.drop(high_null_cols, axis=1)

# Imputar valores nulos
# Categóricas: rellenar con 'None' (muchos NA significan ausencia de la feature)
cat_cols = all_data.select_dtypes(include=['object']).columns
for col in cat_cols:
    all_data[col] = all_data[col].fillna('None')

# Numéricas: rellenar con la mediana
num_cols = all_data.select_dtypes(include=[np.number]).columns
for col in num_cols:
    all_data[col] = all_data[col].fillna(all_data[col].median())

print(f"Nulos restantes: {all_data.isnull().sum().sum()}")
print(f"Shape después de limpieza: {all_data.shape}")

In [ ]:
# Encoding de variables categóricas con LabelEncoder
label_encoders = {}
cat_cols = all_data.select_dtypes(include=['object']).columns

for col in cat_cols:
    le = LabelEncoder()
    all_data[col] = le.fit_transform(all_data[col].astype(str))
    label_encoders[col] = le

print(f"Variables categóricas codificadas: {len(cat_cols)}")
print(f"Shape final: {all_data.shape}")
all_data.head()

## 6. Split train/test y validación

In [ ]:
# Separar de nuevo en train y test
X = all_data[:ntrain]
X_test_final = all_data[ntrain:]

# Split para validación
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test_final (Kaggle): {X_test_final.shape}")

## 7. Modelo 1: Random Forest

In [ ]:
# Random Forest
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Predicciones
rf_train_pred = rf.predict(X_train)
rf_val_pred = rf.predict(X_val)

# Métricas (en escala log)
print("=== Random Forest ===")
print(f"Train RMSE (log): {np.sqrt(mean_squared_error(y_train, rf_train_pred)):.4f}")
print(f"Val RMSE (log):   {np.sqrt(mean_squared_error(y_val, rf_val_pred)):.4f}")
print(f"Val MAE (log):    {mean_absolute_error(y_val, rf_val_pred):.4f}")
print(f"Val R²:           {r2_score(y_val, rf_val_pred):.4f}")

# Cross-validation
rf_cv = cross_val_score(rf, X, y, cv=5, scoring='neg_root_mean_squared_error')
print(f"CV RMSE (log):    {-rf_cv.mean():.4f} (+/- {rf_cv.std():.4f})")

In [ ]:
# Feature importance - Random Forest
rf_importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
rf_importances.head(20).plot(kind='bar')
plt.title('Top 20 Feature Importances - Random Forest')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

## 8. Modelo 2: Gradient Boosting

In [ ]:
# Gradient Boosting
gb = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=3,
    subsample=0.8,
    max_features='sqrt',
    random_state=42
)

gb.fit(X_train, y_train)

# Predicciones
gb_train_pred = gb.predict(X_train)
gb_val_pred = gb.predict(X_val)

# Métricas
print("=== Gradient Boosting ===")
print(f"Train RMSE (log): {np.sqrt(mean_squared_error(y_train, gb_train_pred)):.4f}")
print(f"Val RMSE (log):   {np.sqrt(mean_squared_error(y_val, gb_val_pred)):.4f}")
print(f"Val MAE (log):    {mean_absolute_error(y_val, gb_val_pred):.4f}")
print(f"Val R²:           {r2_score(y_val, gb_val_pred):.4f}")

# Cross-validation
gb_cv = cross_val_score(gb, X, y, cv=5, scoring='neg_root_mean_squared_error')
print(f"CV RMSE (log):    {-gb_cv.mean():.4f} (+/- {gb_cv.std():.4f})")

In [ ]:
# Feature importance - Gradient Boosting
gb_importances = pd.Series(gb.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
gb_importances.head(20).plot(kind='bar', color='green')
plt.title('Top 20 Feature Importances - Gradient Boosting')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

## 9. Comparación de modelos

In [ ]:
# Comparación de modelos
results = pd.DataFrame({
    'Modelo': ['Random Forest', 'Gradient Boosting'],
    'Val RMSE (log)': [
        np.sqrt(mean_squared_error(y_val, rf_val_pred)),
        np.sqrt(mean_squared_error(y_val, gb_val_pred))
    ],
    'Val R²': [
        r2_score(y_val, rf_val_pred),
        r2_score(y_val, gb_val_pred)
    ],
    'CV RMSE (log)': [
        -rf_cv.mean(),
        -gb_cv.mean()
    ]
})
print(results.to_string(index=False))

# Gráfico de comparación
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicciones vs Real - Random Forest
axes[0].scatter(y_val, rf_val_pred, alpha=0.5)
axes[0].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')
axes[0].set_xlabel('Real (log)')
axes[0].set_ylabel('Predicción (log)')
axes[0].set_title('Random Forest: Predicción vs Real')

# Predicciones vs Real - Gradient Boosting
axes[1].scatter(y_val, gb_val_pred, alpha=0.5, color='green')
axes[1].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')
axes[1].set_xlabel('Real (log)')
axes[1].set_ylabel('Predicción (log)')
axes[1].set_title('Gradient Boosting: Predicción vs Real')

plt.tight_layout()
plt.show()

## 10. Ensemble: Promedio de modelos y generación de submission

In [ ]:
# Re-entrenar con TODOS los datos de train para la submission final
rf_final = RandomForestRegressor(
    n_estimators=300, max_depth=15, min_samples_split=5,
    min_samples_leaf=2, max_features='sqrt', random_state=42, n_jobs=-1
)
gb_final = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4,
    min_samples_split=5, min_samples_leaf=3, subsample=0.8,
    max_features='sqrt', random_state=42
)

rf_final.fit(X, y)
gb_final.fit(X, y)

# Predicciones sobre test
rf_test_pred = rf_final.predict(X_test_final)
gb_test_pred = gb_final.predict(X_test_final)

# Ensemble: promedio ponderado (más peso a Gradient Boosting que suele ser mejor)
ensemble_pred = 0.4 * rf_test_pred + 0.6 * gb_test_pred

# Revertir transformación log
final_predictions = np.expm1(ensemble_pred)

print(f"Predicciones generadas: {len(final_predictions)}")
print(f"Precio medio predicho: ${final_predictions.mean():,.0f}")
print(f"Precio min: ${final_predictions.min():,.0f}")
print(f"Precio max: ${final_predictions.max():,.0f}")

In [ ]:
# Generar CSV de submission
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})

submission.to_csv('submission.csv', index=False)
print("Archivo 'submission.csv' generado correctamente.")
submission.head(10)